# Cathode Candidate Screening

Filters Li-ion cathode candidates from ML-screened data and generates VASP DFT inputs for top candidates.

## 1. Load & Filter

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import pandas as pd
from jarvis.db.figshare import data as jarvis_data
from batterymat.screening_cathode.screen_cathode import run_screening

dft3d_df = pd.DataFrame(jarvis_data("dft_3d"))
li_min_df = pd.read_csv("../average_voltage/Li_min.csv")

candidates = run_screening(li_min_df=li_min_df, dft3d_df=dft3d_df)
print(f"Found {len(candidates)} candidates after filtering")
candidates.head(20)

## 2. Visualize

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
candidates["avg_voltage"].hist(ax=axes[0], bins=30, color="steelblue")
axes[0].set_xlabel("Average Voltage (V)")
axes[0].set_ylabel("Count")
axes[0].set_title("Average Voltage Distribution")
candidates["max_grav_cap"].hist(ax=axes[1], bins=30, color="darkorange")
axes[1].set_xlabel("Max Gravimetric Capacity (mAh/g)")
axes[1].set_title("Capacity Distribution")
candidates["ehull"].hist(ax=axes[2], bins=30, color="seagreen")
axes[2].set_xlabel("E-hull (eV)")
axes[2].set_title("Stability Distribution")
plt.tight_layout()
plt.show()

In [ ]:
sc = plt.scatter(candidates["avg_voltage"], candidates["max_grav_cap"], c=candidates["score"], cmap="viridis", alpha=0.7)
plt.colorbar(sc, label="Composite Score")
plt.xlabel("Average Voltage (V)")
plt.ylabel("Max Gravimetric Capacity (mAh/g)")
plt.title("Cathode Candidates: Voltage vs Capacity")
plt.tight_layout()
plt.show()

## 3. Select Candidates & Generate DFT Inputs

In [ ]:
candidates[["jid", "formula", "avg_voltage", "max_grav_cap", "ehull", "optb88vdw_bandgap", "score"]].head(30)

In [ ]:
selected_jids = [
    # "JVASP-XXXXX",
]

from batterymat.screening_cathode.dft_prep import generate_inputs

created_dirs = generate_inputs(
    jids=selected_jids,
    dft3d_df=dft3d_df,
)

print(f"Created {len(created_dirs)} DFT input sets:")
for d in created_dirs:
    print(f"  {d}")